# 🎯 [PRC07_M01] 무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마 정의 (훈련생 실습용)
### 훈련생 실습 워크북 (Google Colab / Jupyter Student Edition)

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/edunlab/milicontent2/blob/main/uploads/PRC07/prc07_m01_student.ipynb)

---

## 📌 1. 프로젝트 및 미션 개요

| 항목 | 내용 |
| :--- | :--- |
| **과정 분류** | 조달·계약·국산화 (Level 3) |
| **미션 ID** | PRC07_M01 |
| **미션 주제** | 무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마 정의 |
| **소요 시간** | 약 1.5시간 |
| **환경 사양** | Python 3.10+, numpy, pandas, matplotlib, scipy, scikit-learn |

### 📖 프로젝트 목표
무기체계 부품은 해외 의존도가 높고 공급선이 하나뿐이거나 단종(DMSMS) 공고가 나오면 조달이 막히기 쉬운데, 위험 신호를 사람이 문서와 전화로 모으면 국산화 후보를 찾기까지 수개월이 걸린다. 본 프로젝트는 부품 공급망 자료에서 조달 병목과 위험 신호를 찾고 국산화 대체 후보를 자동으로 추천하는 도구를 만드는 과제로, 업무 단계별 병목과 위험 신호 기준, 추천 수정안, 계약 의뢰 근거와 변경 이력, 업무 효과 측정까지 한 흐름으로 구현한다. 교육용 가상 부품 100종과 가상 국가·가상 업체 자료로 공급망 데이터 스키마 정의, 공급업체 자료 정제와 단종 공고 파싱, 룰 기반 조달 위험 스크리닝, 텍스트 유사도와 지식 그래프를 쓴 대체 부품 추천, 붕괴 위험도와 조기경보, 위험 지도 대시보드와 오류 처리, 규격 검토와 승인 HITL 워크플로우, 공급망 차단 스트레스 시험, AAR 가중치 재보정, 통합 패키지까지 단계별로 만든다. 최종 목표는 위험 부품을 조기에 찾아 국산화 후보를 근거와 함께 제시하되, 규격 적합성 판단과 조달 승인은 담당자가 하도록 지원하는 도구를 완성하는 것이다.

> **⚠️ [교육용 가상 데이터셋 원칙]**
> - 본 노트북의 모든 데이터(제원, 로그, 시나리오 등)는 보안 규정을 준수하여 **코드 내 난수 기반 가상 데이터로 생성**됩니다.
> - 실제 부대 위치, 작전 인가, 실제 전장 정보, 군사 기밀 데이터는 일체 포함되지 않습니다.

> **💡 [오프라인 폐쇄망 및 로컬 주피터 환경 안내]**
> - 외부 인터넷망(Colab) 외에 군 인트라넷 또는 로컬 환경에서 실행할 경우, 아래 명령어로 필요한 표준 라이브러리를 사전 설치하십시오:
>   `pip install numpy pandas matplotlib scipy scikit-learn`
> - 인터넷이 제한된 환경의 경우 사전 번들링된 Wheel(`.whl`) 패키지 디렉토리를 통해 설치할 수 있습니다:
>   `pip install --no-index --find-links=./wheels/ -r requirements.txt`

> 📁 **[안내: 프로젝트 표준 산출물 디렉터리 가이드]**
> - 본 프로젝트는 단순한 일회성 코드 실습을 넘어 실제 부대에 배포 가능한 업무 도구(Tool)를 완성하는 과정입니다.
> - 실습 중 작성하는 결과물은 역할에 따라 아래 표준 디렉터리에 자동으로 저장·누적됩니다:
>   - `docs/`: 조달 프로세스 분석서, 스키마 정의서, UAT 보고서, 사용자 매뉴얼 등 행정·기술 문서
>   - `src/`: 데이터 로더, 16종 위험 스크리닝 엔진, AI 추천 모델, 대시보드 UI, HITL 결재 FSM 등 핵심 실행 모듈
>   - `dist/`: 폐쇄망 및 타 부서 배포를 위한 원클릭 설치 스크립트(`setup.sh`) 및 최종 패키지

---


In [ ]:
# ============================================================================
# [Google Colab 환경 설정] 필수 라이브러리 설치 및 임포트
# ============================================================================
!pip install -q numpy pandas matplotlib scipy scikit-learn

import os
import sys
import time
import math
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

# 그래프 영문 폰트 기본 설정 (환경 독립성 보장)
plt.rcParams['font.sans-serif'] = 'DejaVu Sans'
plt.rcParams['axes.unicode_minus'] = False

print(f"실행 환경 준비 완료 (Python {sys.version.split()[0]})")

# ============================================================================
# [산출물 자동 저장 유틸리티] 프로젝트 표준 디렉토리(docs/, src/, dist/) 자동 연계
# ============================================================================
def save_deliverable(filepath: str, content: str):
    """프로젝트 표준 deliverables 디렉토리에 산출물 파일을 자동 생성합니다."""
    dirpath = os.path.dirname(filepath)
    if dirpath:
        os.makedirs(dirpath, exist_ok=True)
    with open(filepath, "w", encoding="utf-8") as f:
        f.write(content.strip() + "\n")
    print(f"📦 [산출물 저장 완료] {filepath}")


---

## 🎯 미션 01. 무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마 정의
**소요시간**: 약 1.5시간

### 📋 미션 배경 및 수행 지침
#### 미션 배경 및 당위성

무기체계 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마를 정의한다. 실제 전술 현장에서는 단순 탐지에 그치지 않고 신속한 위협 판단과 신뢰도 높은 의사결정이 필수적이므로, **무기체계 부품 공급망(단종 부품, 해외 의존도, 리드타임) 자료의 스키마와 입출력 명세를 정의하고, 현행(AS-IS) 국산화 후보 선정 업무를 단계로 나누어 병목을 찾는다.** 과업을 완수하여 오판을 방지하고 작전 대응력을 극대화해야 합니다.

이 문제를 해결하기 위해 본 미션에서는 **부품 식별, 공급망(공급선 수, 해외 의존, 리드타임, 납기 지연), 단종 상태(최종구매 잔여 개월), 조달 속성(재고, 수요, 불량률, 수출 허가)을 묶어 필드 16개의 스키마와 검증 규칙 작성하여 원인을 분석하는** 실습을 체계적으로 수행합니다.

학습자는 실습가이드에 제시된 총 4개 세부 활동(기본 기능 구현 ➔ 결과 지표 검증 ➔ 파라미터 튜닝 및 도전과제)을 순차적으로 진행하며 핵심 판정 기준을 달성합니다.

**이번 결과물:** `docs/prc07_as_is_to_be_spec.md` 형태의 결과물을 완성한다.

### 🔍 평가 및 검증 기준 (PASS 조건)
#### 미션 판정

**평가 구분:** 필수

**핵심 평가영역:** 현행 업무(AS-IS)와 개선 업무(TO-BE) 비교의 명확성

#### 미션 PASS 조건

- 정량 기준: 업무 단계 분해 10단계 이상, 병목 구간 3개소 이상 도출


- 진행 기준: 업무 정의서 및 입출력 데이터 명세 작성 완료


- 검증 방법: 실무 프로세스와의 일치성 검증


#### 채점자 판정 메모

> 단순 코드 실행 여부뿐만 아니라 출력된 수치와 근거 자료가 군 실무 요구사항에 부합하는지 면밀히 확인한다.



In [ ]:
"""무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마를 정의·검증하고, 국산화 후보 선정 업무를 AS-IS/TO-BE 단계로 분해해 병목 구간을 도출한다."""
import json
import re
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ---------------------------------------------------------------------------
# [교육용가상자료] 가상 무기체계 부품·공급망 데이터 생성기 (실제 체계·업체·국가·단가 정보 없음)
# 부품 100종(해외 조달 약 80%)의 공급망 속성과, 국내 부품 카탈로그(과거 승인된 국산 대체품 +
# 설명은 비슷하지만 규격이 맞지 않는 유사 후보)를 함께 만든다.
# ---------------------------------------------------------------------------
COUNTRIES = ["가상국가A", "가상국가B", "가상국가C", "국내"]
CATS = {
    "전원모듈": dict(words=["DC-DC", "컨버터", "절연형", "전원", "정출력"], syn={"컨버터": "변환장치", "절연형": "아이솔레이션", "정출력": "안정화"}, std="MIL-STD-704", iface="단자대"),
    "커넥터": dict(words=["원형", "커넥터", "방수형", "다핀", "나사체결"], syn={"커넥터": "접속자", "방수형": "밀폐형", "나사체결": "스크루결합"}, std="MIL-DTL-38999", iface="원형핀"),
    "유압밸브": dict(words=["유압", "솔레노이드", "밸브", "방향제어", "4포트"], syn={"밸브": "개폐장치", "방향제어": "유로전환", "솔레노이드": "전자식"}, std="SAE-AS5440", iface="플랜지"),
    "베어링": dict(words=["볼", "베어링", "고속", "밀봉형", "회전축"], syn={"베어링": "축받이", "밀봉형": "실드형", "고속": "고회전"}, std="ISO-492", iface="축결합"),
    "광학렌즈": dict(words=["적외선", "렌즈", "게르마늄", "반사방지", "집광"], syn={"렌즈": "광학창", "집광": "초점형성", "반사방지": "AR코팅"}, std="MIL-PRF-13830", iface="마운트"),
    "자이로센서": dict(words=["광섬유", "자이로", "각속도", "센서", "관성"], syn={"센서": "감지기", "각속도": "회전율", "관성": "INS용"}, std="IEEE-952", iface="직렬통신"),
    "RF증폭기": dict(words=["RF", "전력증폭기", "광대역", "모듈", "송신"], syn={"전력증폭기": "파워앰프", "송신": "출력단", "광대역": "와이드밴드"}, std="MIL-STD-461", iface="SMA"),
    "제어보드": dict(words=["FPGA", "제어보드", "임베디드", "연산", "입출력"], syn={"제어보드": "컨트롤카드", "연산": "신호처리", "임베디드": "내장형"}, std="VITA-46", iface="백플레인"),
}
V_OPTS, W_OPTS = [12, 24, 28], [25, 50, 100]
VENDORS = ["가상정밀(주)", "가상전자(주)", "가상유공압(주)", "가상광학(주)", "가상테크(주)", "가상시스템(주)"]


def describe(cat, words, s):
    # [TODO : 학생 실습 1] 무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def _sig(x):
    # [TODO : 학생 실습 1] 무기체계 핵심 부품 공급망(단종 부품, 해외 의존도, 리드타임) 데이터셋 스키마 정의 관련 핵심 로직을 구현하세요.
    # 힌트: 상단 주석 및 데이터 스키마/지침을 참고하여 작성합니다.
    # <<< 여기에 코드를 작성하세요 >>>
    pass  # 작성 후 pass를 제거하세요.
def make_parts(n=100, seed=7):
    """부품 목록(공급망 속성 포함)과 국내 카탈로그를 만든다. 반환: (parts DataFrame, catalog DataFrame)"""
    rng = np.random.default_rng(seed)
    cats, rows, cat_rows = list(CATS), [], []
    for i in range(n):
        cat = cats[i % len(cats)]; c = CATS[cat]
        spec = {"v": int(rng.choice(V_OPTS)), "w": int(rng.choice(W_OPTS)), "tmin": int(rng.choice([-55, -40])),
                "tmax": int(rng.choice([85, 125])), "size_mm": float(round(rng.uniform(20, 120), 1)), "std": c["std"]}
        words = [str(x) for x in rng.permutation(c["words"])[:4]]
        country = str(rng.choice(COUNTRIES, p=[0.35, 0.3, 0.15, 0.2]))
        overseas = country != "국내"
        n_sup = int(rng.choice([1, 2, 3], p=[0.45, 0.35, 0.2])) if overseas else int(rng.choice([1, 2, 3]))
        lead = float(round(rng.gamma(3, 45 if overseas else 25) + 20))
        eol = str(rng.choice(["정상", "단종예고", "단종"], p=[0.7, 0.2, 0.1]))
        crit = str(rng.choice(["A", "B", "C"], p=[0.3, 0.4, 0.3]))
        z = (-2.6 + 1.1 * (n_sup == 1) + 0.9 * overseas + 0.006 * (lead - 100) + 1.2 * (eol == "단종예고")
             + 2.0 * (eol == "단종") + rng.normal(0, 0.6))
        p_late = float(np.clip(_sig(z) * 0.8 + 0.05, 0, 0.9))
        late = int(rng.binomial(12, p_late))
        demand = int(rng.integers(20, 400))
        rows.append({"part_no": f"P-{i + 1:04d}", "category": cat, "desc": describe(cat, words, spec), **spec,
                     "iface": c["iface"], "country": country, "overseas": overseas, "n_suppliers": n_sup,
                     "lead_time_days": lead, "orders_12m": 12, "late_orders": late,
                     "avg_delay_days": float(round(late and rng.gamma(2, 12 + 20 * p_late), 1)),
                     "eol_status": eol, "ltb_months": int(rng.integers(1, 25)) if eol == "단종예고" else None,
                     "price_change_pct": float(round(rng.normal(4 + 12 * p_late, 8), 1)), "criticality": crit,
                     "annual_demand": demand, "stock_qty": int(rng.integers(0, demand)),
                     "defect_rate": float(round(abs(rng.normal(0.8 + 2 * p_late, 0.8)), 2)),
                     "export_license": bool(overseas and rng.random() < 0.35),
                     "disrupted_next12m": bool(rng.random() < _sig(z + 0.4))})
        if not overseas:
            continue
        # 과거 승인된 국산 대체품(정답): 동의어로 서술, 규격 호환(전력 여유 허용, 일부는 크기 차이가 커 장착 검토 필요)
        eq_words = [c["syn"].get(w, w) for w in words]
        hard = rng.random() < 0.15                                   # 서술이 많이 다른 사례
        if hard:
            eq_words = [c["syn"].get(w, w) for w in c["words"] if w not in words][:1] + eq_words[1:3]
        eq = {**spec, "w": spec["w"] if rng.random() < 0.7 else W_OPTS[min(W_OPTS.index(spec["w"]) + 1, 2)],
              "size_mm": float(round(spec["size_mm"] * (rng.uniform(1.11, 1.15) if rng.random() < 0.12 else rng.uniform(0.95, 1.05)), 1))}
        cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, eq_words, eq), **eq,
                         "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": f"P-{i + 1:04d}", "kind": "equivalent"})
        # 유사 오답 후보: 설명은 거의 같지만 전압이 다르거나(5V/48V) 상용 온도 등급(0~70C)
        for kind in ("volt_mismatch", "temp_mismatch"):
            d = dict(spec)
            if kind == "volt_mismatch":
                d["v"] = int(rng.choice([5, 48]))                           # 체계에서 쓰지 않는 전압
            else:
                d["tmin"], d["tmax"] = 0, 70
            d["size_mm"] = float(round(spec["size_mm"] * rng.uniform(0.95, 1.05), 1))
            cat_rows.append({"cand_no": f"D-{len(cat_rows) + 1:04d}", "category": cat, "desc": describe(cat, words, d), **d,
                             "iface": c["iface"], "vendor": str(rng.choice(VENDORS)), "approved_for": None, "kind": kind})
    parts = pd.DataFrame(rows)
    catalog = pd.DataFrame(cat_rows).sample(frac=1, random_state=seed).reset_index(drop=True)
    return parts, catalog
# ---------------------------------------------------------------------------

# 1) 스키마 정의: 부품 식별, 공급망(해외 의존도·공급선 수·리드타임·지연 이력), 단종(DMSMS), 조달 속성
SCHEMA = {
    "part_no": {"type": str, "pattern": r"^P-\d{4}$", "desc": "부품 번호"},
    "category": {"type": str, "enum": list(CATS), "desc": "부품 범주"},
    "country": {"type": str, "enum": COUNTRIES, "desc": "주 공급 국가(가상)"},
    "n_suppliers": {"type": int, "min": 1, "max": 10, "desc": "공급선 수"},
    "lead_time_days": {"type": (int, float), "min": 1, "max": 1000, "desc": "조달 리드타임(일)"},
    "orders_12m": {"type": int, "min": 0, "max": 100, "desc": "최근 12개월 발주 건수"},
    "late_orders": {"type": int, "min": 0, "max": 100, "desc": "납기 지연 건수"},
    "avg_delay_days": {"type": (int, float), "min": 0, "max": 365, "desc": "평균 지연일"},
    "eol_status": {"type": str, "enum": ["정상", "단종예고", "단종"], "desc": "단종 상태"},
    "ltb_months": {"type": (int, float), "min": 0, "max": 60, "optional": True, "desc": "최종구매(LTB)까지 남은 개월"},
    "price_change_pct": {"type": (int, float), "min": -100, "max": 500, "desc": "전년 대비 단가 변동(%)"},
    "criticality": {"type": str, "enum": ["A", "B", "C"], "desc": "체계 중요도"},
    "annual_demand": {"type": int, "min": 0, "max": 100000, "desc": "연간 소요량"},
    "stock_qty": {"type": int, "min": 0, "max": 100000, "desc": "현 재고"},
    "defect_rate": {"type": (int, float), "min": 0, "max": 100, "desc": "입고 불량률(%)"},
    "export_license": {"type": bool, "desc": "수출 허가 필요 여부"},
}


def check(name, v, r):
    if v is None or (isinstance(v, float) and np.isnan(v)):
        return [] if r.get("optional") else [f"{name}: 필수 값 누락"]
    if not isinstance(v, r["type"]) or (isinstance(v, bool) and r["type"] is not bool):
        return [f"{name}: 자료형 오류({type(v).__name__})"]
    errs = []
    if "enum" in r and v not in r["enum"]:
        errs.append(f"{name}: 허용되지 않은 값 '{v}'")
    if "min" in r and v < r["min"]:
        errs.append(f"{name}: 최소값 {r['min']} 미만")
    if "max" in r and v > r["max"]:
        errs.append(f"{name}: 최대값 {r['max']} 초과")
    if "pattern" in r and not re.match(r["pattern"], v):
        errs.append(f"{name}: 형식 불일치 '{v}'")
    return errs


def validate(rec):
    errs = [f"{k}: 필드 누락" for k in SCHEMA if k not in rec]
    for k, r in SCHEMA.items():
        if k in rec:
            errs += check(k, rec[k], r)
    if not errs and rec["late_orders"] > rec["orders_12m"]:           # 필드 간 논리 규칙
        errs.append("late_orders: 발주 건수보다 지연 건수가 많음")
    if not errs and rec["eol_status"] == "단종예고" and rec["ltb_months"] is None:
        errs.append("ltb_months: 단종예고 부품은 LTB 개월 필수")
    return errs


def to_record(row):
    rec = {k: (row[k].item() if hasattr(row[k], "item") else row[k]) for k in SCHEMA}
    rec["ltb_months"] = None if pd.isna(rec["ltb_months"]) else int(rec["ltb_months"])
    return rec


parts, _ = make_parts(100, seed=7)
records = [to_record(r) for _, r in parts.iterrows()]
ok = sum(not validate(r) for r in records)
print(f"[스키마 검증] 부품 {len(records)}건 중 통과 {ok}건, 오류 {len(records) - ok}건 / 필드 {len(SCHEMA)}개")
print(f"해외 의존 {sum(r['country'] != '국내' for r in records)}건, 단일 공급선 {sum(r['n_suppliers'] == 1 for r in records)}건, "
      f"단종예고·단종 {sum(r['eol_status'] != '정상' for r in records)}건, 평균 리드타임 {np.mean([r['lead_time_days'] for r in records]):.0f}일")

# 2) 오류 주입 시험: 형식·허용값·범위·자료형·논리 규칙 위반 6종
bad = json.loads(json.dumps(records[2], ensure_ascii=False))
bad.update({"part_no": "P12", "country": "미상", "lead_time_days": -5, "n_suppliers": "2"})
bad2 = dict(records[5], late_orders=15, orders_12m=12)
bad3 = dict(records[0], eol_status="단종예고", ltb_months=None)
print("\n[오류 주입 결과]")
for e in validate(bad) + validate(bad2) + validate(bad3):
    print(" -", e)

# 3) AS-IS 업무 단계 분해: [교육용가상자료] 국산화 후보 선정 30건의 단계별 소요일(작업 + 대기)
rng = np.random.default_rng(3)
STEPS = [("체계 부품목록(BOM) 수집", 2, 1), ("공급업체 자료 요청·회신", 3, 9), ("단종(DMSMS) 공고 수집·확인", 2, 6),
         ("조달 이력·납기 지연 수작업 집계", 5, 2), ("해외 의존도 분류", 2, 1), ("위험 부품 선정 회의", 1, 7),
         ("국내 업체 탐색(전화·자료)", 6, 8), ("규격 비교표 작성", 5, 2), ("기술 검토 의뢰", 1, 10),
         ("국산화 후보 보고서 작성", 3, 1), ("결재(승인)", 1, 5), ("계약 의뢰 준비", 2, 2)]
TOBE = {1: ("표준 서식 자동 수집", 0.3), 2: ("공고 자동 파싱(미션 2)", 0.2), 3: ("룰 엔진 자동 집계(미션 3)", 0.1),
        4: ("자동 분류(미션 3)", 0.1), 6: ("대체품 자동 추천(미션 4)", 0.3), 7: ("규격 자동 대조(미션 4)", 0.2),
        9: ("보고서 초안 자동 생성(미션 10)", 0.4)}
rows = []
for i, (name, work, wait) in enumerate(STEPS):
    w = rng.gamma(4, work / 4, 30); q = rng.gamma(2, wait / 2, 30)
    f = TOBE.get(i, ("유지(사람 판단)", 1.0))
    rows.append({"단계": f"{i + 1:02d}. {name}", "작업(일)": w.mean(), "대기(일)": q.mean(), "AS-IS(일)": w.mean() + q.mean(),
                 "TO-BE 방식": f[0], "TO-BE(일)": w.mean() * f[1] + q.mean() * (f[1] if f[1] < 1 else 1.0)})
proc = pd.DataFrame(rows)
share = proc["AS-IS(일)"] / proc["AS-IS(일)"].sum()
proc["병목"] = np.where((share >= share.sort_values().iloc[-3]) | (proc["대기(일)"] / proc["AS-IS(일)"] > 0.75), "병목", "")
print("\n[AS-IS / TO-BE 업무 단계 비교]")
print(proc.round(1).to_string(index=False))
n_bn = int((proc["병목"] == "병목").sum())
asis, tobe = proc["AS-IS(일)"].sum(), proc["TO-BE(일)"].sum()
print(f"\n업무 단계 {len(proc)}개(목표 10개 이상: {'달성' if len(proc) >= 10 else '미달'}), 병목 {n_bn}개소(목표 3개소 이상: {'달성' if n_bn >= 3 else '미달'})")
print(f"후보 선정 1건 소요: AS-IS {asis:.1f}일 -> TO-BE {tobe:.1f}일 ({1 - tobe / asis:.0%} 단축 예상). 결재·기술 검토·회의는 사람이 수행한다.")

# 4) 입출력 데이터 명세 요약
print("\n[입출력 명세] 입력: 부품 공급망 레코드(위 16개 필드) / 출력: 위험 등급, 국산화 후보 Top-3, 추천 근거, 결재 이력")

plt.figure(figsize=(9, 5))
y = np.arange(len(proc))
plt.barh(y - 0.2, proc["AS-IS(일)"], 0.4, label="AS-IS", color="tab:gray")
plt.barh(y + 0.2, proc["TO-BE(일)"], 0.4, label="TO-BE", color="tab:blue")
plt.yticks(y, [f"step {i + 1}{' *' if b else ''}" for i, b in enumerate(proc["병목"])])
plt.gca().invert_yaxis(); plt.xlabel("days per case"); plt.legend()
plt.title("Localization candidate process: AS-IS vs TO-BE (* = bottleneck, synthetic)")
plt.tight_layout(); plt.show()


<details>
<summary><b>💡 [참고] 미션 01 예상 실행 결과 (Output)</b></summary>

```text
[스키마 검증] 부품 100건 중 통과 100건, 오류 0건 / 필드 16개
해외 의존 85건, 단일 공급선 42건, 단종예고·단종 37건, 평균 리드타임 145일

[오류 주입 결과]
 - part_no: 형식 불일치 'P12'
 - country: 허용되지 않은 값 '미상'
 - n_suppliers: 자료형 오류(str)
 - lead_time_days: 최소값 1 미만
 - late_orders: 발주 건수보다 지연 건수가 많음
 - ltb_months: 단종예고 부품은 LTB 개월 필수

[AS-IS / TO-BE 업무 단계 비교]
                    단계  작업(일)  대기(일)  AS-IS(일)            TO-BE 방식  TO-BE(일) 병목
   01. 체계 부품목록(BOM) 수집    2.0    1.0       3.0           유지(사람 판단)       3.0   
     02. 공급업체 자료 요청·회신    3.9    8.5      12.4         표준 서식 자동 수집       3.7 병목
03. 단종(DMSMS) 공고 수집·확인    2.1    6.6       8.7      공고 자동 파싱(미션 2)       1.7 병목
04. 조달 이력·납기 지연 수작업 집계    5.9    2.0       7.9    룰 엔진 자동 집계(미션 3)       0.8   
         05. 해외 의존도 분류    2.4    1.1       3.5         자동 분류(미션 3)       0.4   
       06. 위험 부품 선정 회의    1.0    7.2       8.3           유지(사람 판단)       8.3 병목
   07. 국내 업체 탐색(전화·자료)    5.3    7.5      12.8     대체품 자동 추천(미션 4)       3.8 병목
         08. 규격 비교표 작성    4.7    2.0       6.7      규격 자동 대조(미션 4)       1.3   
          09. 기술 검토 의뢰    1.1    9.6      10.7           유지(사람 판단)      10.7 병목
     10. 국산화 후보 보고서 작성    3.1    1.2       4.3 보고서 초안 자동 생성(미션 10)       1.7   
            11. 결재(승인)    0.9    5.7       6.6           유지(사람 판단)       6.6 병목
          12. 계약 의뢰 준비    2.1    2.1       4.1           유지(사람 판단)       4.1   

업무 단계 12개(목표 10개 이상: 달성), 병목 6개소(목표 3개소 이상: 달성)
후보 선정 1건 소요: AS-IS 89.1일 -> TO-BE 46.3일 (48% 단축 예상). 결재·기술 검토·회의는 사람이 수행한다.

[입출력 명세] 입력: 부품 공급망 레코드(위 16개 필드) / 출력: 위험 등급, 국산화 후보 Top-3, 추천 근거, 결재 이력
```
</details>
